In [1]:
import os
import re
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from IPython.display import clear_output

# Tell PIL to be tolerant of slightly corrupted/truncated JPEGs
ImageFile.LOAD_TRUNCATED_IMAGES = True

# 16 development phases as defined in the paper
PHASES = [
    "tPB2", "tPNa", "tPNf", "t2", "t3", "t4", "t5", "t6",
    "t7", "t8", "t9+", "tM", "tSB", "tB", "tEB", "tHB",
]
PHASE_TO_IDX = {phase: i for i, phase in enumerate(PHASES)}

# Local paths (keeps Path math consistent across cells)
BASE_DIR = Path.cwd()
IMG_DIR = BASE_DIR / "embryo_dataset"
ANNO_DIR = BASE_DIR / "embryo_dataset_annotations"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


assert IMG_DIR.exists(), f"IMG_DIR not found: {IMG_DIR}"
assert ANNO_DIR.exists(), f"ANNO_DIR not found: {ANNO_DIR}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python Version: {os.sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Active Device: {device}")
print(f"torch.version.cuda: {torch.version.cuda}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")

if device.type == "cuda":
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")

Python Version: 3.14.0
PyTorch Version: 2.11.0+cpu
Active Device: cpu
torch.version.cuda: None
torch.cuda.is_available(): False


In [30]:
RUN_RE = re.compile(r"_RUN(\d+)\.(?:jpe?g|png)$", re.IGNORECASE)

def _run_number(filename: str) -> int | None:
    m = RUN_RE.search(filename)
    return int(m.group(1)) if m else None


class EmbryoDataset(Dataset):
    def __init__(self, img_dir: Path, anno_dir: Path, video_ids, transform=None, sample_rate: int = 1):
        self.img_dir = Path(img_dir)
        self.anno_dir = Path(anno_dir)
        self.video_ids = list(video_ids)
        self.transform = transform
        self.sample_rate = int(sample_rate)
        self.samples = self._prepare_samples()

    def _prepare_samples(self):
        samples = []
        for vid in self.video_ids:
            csv_path = self.anno_dir / f"{vid}_phases.csv"
            vid_folder = self.img_dir / vid

            if (not csv_path.exists()) or (not vid_folder.exists()):
                continue

            # Build map: RUN number -> absolute image path
            run_to_path = {}
            for f in os.listdir(vid_folder):
                rn = _run_number(f)
                if rn is not None:
                    run_to_path[rn] = str(vid_folder / f)

            if not run_to_path:
                continue

            # CSV has NO header: phase,start_run,end_run (both inclusive)
            df = pd.read_csv(csv_path, header=None)
            if df.shape[1] < 3:
                continue

            for _, row in df.iterrows():
                phase_name = str(row.iloc[0]).strip()
                if phase_name not in PHASE_TO_IDX:
                    continue
                label = PHASE_TO_IDX[phase_name]
                start_run = int(row.iloc[1])
                end_run = int(row.iloc[2])

                # Sample by RUN number (avoids lexicographic sorting bugs)
                for run in range(start_run, end_run + 1, self.sample_rate):
                    img_path = run_to_path.get(run)
                    if img_path is not None:
                        samples.append((img_path, label))

        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

In [31]:
# For Inception v3, 299x299 is the standard input size
IMAGE_SIZE = 299

# --- 1. Data Transforms ---
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# --- 2. The 70/15/15 Video-Level Split ---
all_vids = [d for d in os.listdir(IMG_DIR) if (IMG_DIR / d).is_dir()]
train_vids, temp_vids = train_test_split(all_vids, test_size=0.3, random_state=42)
val_vids, test_vids = train_test_split(temp_vids, test_size=0.5, random_state=42)
print(f"Video Split -> Train: {len(train_vids)} | Val: {len(val_vids)} | Test: {len(test_vids)}")

# --- 3. Build Datasets & Loaders ---
SAMPLE_RATE = 10  # set to 1 for final training
# InceptionV3 @ 299px can OOM at 32 on 8GB GPUs; start safer and scale up if you want
BATCH_SIZE = 16 if device.type == "cuda" else 4

train_dataset = EmbryoDataset(IMG_DIR, ANNO_DIR, train_vids, transform=train_transforms, sample_rate=SAMPLE_RATE)
val_dataset = EmbryoDataset(IMG_DIR, ANNO_DIR, val_vids, transform=val_test_transforms, sample_rate=SAMPLE_RATE)
test_dataset = EmbryoDataset(IMG_DIR, ANNO_DIR, test_vids, transform=val_test_transforms, sample_rate=SAMPLE_RATE)
print(f"Images Loaded -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Windows + notebooks can be flaky with multi-process DataLoader; start with 0 workers
NUM_WORKERS = 0 if os.name == "nt" else 2
PIN_MEMORY = (device.type == "cuda")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print("Success: All DataLoaders defined.")

Video Split -> Train: 492 | Val: 106 | Test: 106
Images Loaded -> Train: 23376 | Val: 5115 | Test: 5021
Success: All DataLoaders defined.


In [32]:
# Optional: offline Inception weights fallback (useful if download.pytorch.org is blocked)
INCEPTIONV3_LOCAL_WEIGHTS = WEIGHTS_DIR / "inception_v3_google-0cc3c7bd.pth"


def _load_inception_v3_backbone() -> nn.Module:
    try:
        return models.inception_v3(weights="DEFAULT")
    except Exception as e:
        if INCEPTIONV3_LOCAL_WEIGHTS.exists():
            backbone = models.inception_v3(weights=None)
            state_dict = torch.load(str(INCEPTIONV3_LOCAL_WEIGHTS), map_location="cpu")
            backbone.load_state_dict(state_dict)
            print(f"Loaded Inception v3 weights from: {INCEPTIONV3_LOCAL_WEIGHTS}")
            return backbone
        raise RuntimeError(
            "Could not load pretrained Inception v3 weights.\n"
            "- If you have internet, re-run this cell (it will download once).\n"
            f"- If blocked/offline, place weights at: {INCEPTIONV3_LOCAL_WEIGHTS}\n"
            f"Original error: {repr(e)}"
        ) from e


class EmbryoMobileNet(nn.Module):
    def __init__(self):
        super(EmbryoMobileNet, self).__init__()
        try:
            self.backbone = models.mobilenet_v2(weights="DEFAULT")
        except Exception as e:
            print(f"[MobileNet] Could not load pretrained weights, using random init. Reason: {repr(e)}")
            self.backbone = models.mobilenet_v2(weights=None)
        num_ftrs = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.2), nn.Linear(num_ftrs, 128), nn.ReLU(), nn.Linear(128, 16)
        )
    def forward(self, x):
        return self.backbone(x)


class EmbryoGoogLeNet(nn.Module):
    def __init__(self):
        super(EmbryoGoogLeNet, self).__init__()
        try:
            self.backbone = models.googlenet(weights="DEFAULT")
        except Exception as e:
            print(f"[GoogLeNet] Could not load pretrained weights, using random init. Reason: {repr(e)}")
            self.backbone = models.googlenet(weights=None)
        self.backbone.aux_logits = False
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.2), nn.Linear(num_ftrs, 128), nn.ReLU(), nn.Linear(128, 16)
        )
    def forward(self, x):
        return self.backbone(x)


class EmbryoInceptionV3(nn.Module):
    def __init__(self):
        super(EmbryoInceptionV3, self).__init__()
        self.backbone = _load_inception_v3_backbone()
        self.backbone.aux_logits = False
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.2), nn.Linear(num_ftrs, 128), nn.ReLU(), nn.Linear(128, 16)
        )
    def forward(self, x):
        return self.backbone(x)


class EmbryoVGG(nn.Module):
    def __init__(self, use_vgg19=False):
        super(EmbryoVGG, self).__init__()
        try:
            self.backbone = models.vgg19(weights="DEFAULT") if use_vgg19 else models.vgg16(weights="DEFAULT")
        except Exception as e:
            which = "VGG19" if use_vgg19 else "VGG16"
            print(f"[{which}] Could not load pretrained weights, using random init. Reason: {repr(e)}")
            self.backbone = models.vgg19(weights=None) if use_vgg19 else models.vgg16(weights=None)
        for param in self.backbone.features.parameters():
            param.requires_grad = False
        num_ftrs = self.backbone.classifier[6].in_features
        self.backbone.classifier[6] = nn.Linear(num_ftrs, 16)
    def forward(self, x):
        return self.backbone(x)

In [33]:
def train_and_evaluate(model, model_name, alpha_name, alpha, num_epochs=10):
    print(f"\n🚀 STARTING: {model_name} | {alpha_name} (Alpha={alpha}) 🚀")
    
    ce_loss_fn = nn.CrossEntropyLoss()
    mse_loss_fn = nn.MSELoss()
    phase_indices = torch.arange(16, dtype=torch.float32, device=device)
    
    # Filter out frozen parameters so the optimizer doesn't waste memory tracking them
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(trainable_params, lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    checkpoint_path = str(OUTPUT_DIR / f"{model_name}_{alpha_name}.pth")
    
    history = {'train_loss': [], 'train_exact_acc': [], 'train_tol_acc': [], 'val_loss': [], 'val_exact_acc': [], 'val_tol_acc': []}
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        model.train()
        running_loss, exact_correct, tol_correct, total = 0.0, 0, 0, 0
        
        for i, (images, labels) in enumerate(train_loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            labels_long = labels.long()
            labels_float = labels.float()
            
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            
            loss_ce = ce_loss_fn(logits, labels_long)
            probs = F.softmax(logits, dim=1)
            expected_phase = torch.sum(probs * phase_indices, dim=1)
            loss_mse = mse_loss_fn(expected_phase, labels_float)
            
            # The Custom Loss Toggle
            loss = (alpha * loss_ce) + ((1 - alpha) * loss_mse)
            loss.backward()
            optimizer.step()
            
            preds = torch.argmax(logits, dim=1)
            batch_size = labels_long.size(0)
            total += batch_size
            exact_correct += (preds == labels_long).sum().item()
            tol_correct += (torch.abs(preds - labels_long) <= 1).sum().item()
            running_loss += loss.item()
            
            if (i + 1) % 100 == 0:
                print(f"[{model_name}] Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
            
            # STRICT LOCAL VRAM WIPE
            del images, labels, labels_long, labels_float, logits, probs, expected_phase, loss, preds
        
        history['train_loss'].append(running_loss / max(1, len(train_loader)))
        history['train_exact_acc'].append(exact_correct / max(1, total))
        history['train_tol_acc'].append(tol_correct / max(1, total))
        
        # Validation Phase
        model.eval()
        val_loss, val_exact, val_tol, val_total = 0.0, 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                labels_long = labels.long()
                labels_float = labels.float()
                
                logits = model(images)
                loss_ce = ce_loss_fn(logits, labels_long)
                probs = F.softmax(logits, dim=1)
                expected_phase = torch.sum(probs * phase_indices, dim=1)
                loss_mse = mse_loss_fn(expected_phase, labels_float)
                loss = (alpha * loss_ce) + ((1 - alpha) * loss_mse)
                
                preds = torch.argmax(logits, dim=1)
                batch_size = labels_long.size(0)
                val_total += batch_size
                val_exact += (preds == labels_long).sum().item()
                val_tol += (torch.abs(preds - labels_long) <= 1).sum().item()
                val_loss += loss.item()
                
                del images, labels, labels_long, labels_float, logits, probs, expected_phase, loss, preds
        
        current_val_loss = val_loss / max(1, len(val_loader))
        history['val_loss'].append(current_val_loss)
        history['val_exact_acc'].append(val_exact / max(1, val_total))
        history['val_tol_acc'].append(val_tol / max(1, val_total))
        scheduler.step(current_val_loss)
        
        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'history': history, 'best_val_loss': best_val_loss}, checkpoint_path)
        
        print(f"[{model_name}] End of Epoch {epoch+1} | Train Tol: {history['train_tol_acc'][-1]:.2%} | Val Tol: {history['val_tol_acc'][-1]:.2%}")
    
    # Plot & Save Individual Graph
    plt.figure(figsize=(18, 5))
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Train', color='red', marker='o')
    plt.plot(history['val_loss'], label='Val', color='darkred', marker='x', linestyle='--')
    plt.title(f'{model_name} ({alpha_name}) Loss')
    plt.legend()
    
    plt.subplot(1, 3, 2)
    plt.plot(history['train_exact_acc'], label='Train Exact', color='blue', marker='o')
    plt.plot(history['val_exact_acc'], label='Val Exact', color='darkblue', marker='x', linestyle='--')
    plt.title(f'{model_name} Exact Accuracy')
    plt.legend()
    
    plt.subplot(1, 3, 3)
    plt.plot(history['train_tol_acc'], label='Train Tol (±1)', color='green', marker='o')
    plt.plot(history['val_tol_acc'], label='Val Tol (±1)', color='darkgreen', marker='x', linestyle='--')
    plt.title(f'{model_name} Tolerance Acc')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / f"{model_name}_{alpha_name}_results.png"))
    plt.close()

In [34]:
alphas_to_test = [
    ("Baseline_CE", 1.0),
    ("Hybrid_MSE", 0.5),
]

models_to_train = [
    ("MobileNet", EmbryoMobileNet),
    ("GoogLeNet", EmbryoGoogLeNet),
    ("InceptionV3", EmbryoInceptionV3),
    ("VGG16", lambda: EmbryoVGG(use_vgg19=False)),
    ("VGG19", lambda: EmbryoVGG(use_vgg19=True)),
]

NUM_EPOCHS = 10
print(f"Starting Tournament: {len(models_to_train) * len(alphas_to_test)} total runs scheduled.")

for model_name, ModelClass in models_to_train:
    for alpha_name, ALPHA in alphas_to_test:
        # 1. Initialize a fresh model
        model = ModelClass().to(device)
        # 2. Run the heavy math
        train_and_evaluate(model, model_name, alpha_name, ALPHA, num_epochs=NUM_EPOCHS)
        # 3. Deep Clean GPU to protect the next run
        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

print("\nLOCAL TRAINING COMPLETED SUCCESSFULLY!")

Starting Tournament: 10 total runs scheduled.


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\Swarup/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 23.8MB/s]



🚀 STARTING: MobileNet | Baseline_CE (Alpha=1.0) 🚀
[MobileNet] Epoch [1/10], Step [100/1461], Loss: 1.8164
[MobileNet] Epoch [1/10], Step [200/1461], Loss: 1.7479
[MobileNet] Epoch [1/10], Step [300/1461], Loss: 1.2448
[MobileNet] Epoch [1/10], Step [400/1461], Loss: 1.6184
[MobileNet] Epoch [1/10], Step [500/1461], Loss: 0.9623
[MobileNet] Epoch [1/10], Step [600/1461], Loss: 1.4164
[MobileNet] Epoch [1/10], Step [700/1461], Loss: 1.2106
[MobileNet] Epoch [1/10], Step [800/1461], Loss: 1.3991
[MobileNet] Epoch [1/10], Step [900/1461], Loss: 1.4806
[MobileNet] Epoch [1/10], Step [1000/1461], Loss: 1.2270
[MobileNet] Epoch [1/10], Step [1100/1461], Loss: 0.9302
[MobileNet] Epoch [1/10], Step [1200/1461], Loss: 1.3481
[MobileNet] Epoch [1/10], Step [1300/1461], Loss: 1.1596
[MobileNet] Epoch [1/10], Step [1400/1461], Loss: 1.3413
[MobileNet] End of Epoch 1 | Train Tol: 75.26% | Val Tol: 83.34%
[MobileNet] Epoch [2/10], Step [100/1461], Loss: 1.2965
[MobileNet] Epoch [2/10], Step [200/146

Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to C:\Users\Swarup/.cache\torch\hub\checkpoints\googlenet-1378be20.pth
c:\Users\Swarup\miniforge3\envs\embryo_env\lib\site-packages\torchvision\models\googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


[GoogLeNet] Could not load pretrained weights, using random init. Reason: URLError(ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

🚀 STARTING: GoogLeNet | Baseline_CE (Alpha=1.0) 🚀
[GoogLeNet] Epoch [1/10], Step [100/1461], Loss: 2.0401
[GoogLeNet] Epoch [1/10], Step [200/1461], Loss: 2.0057
[GoogLeNet] Epoch [1/10], Step [300/1461], Loss: 1.8689
[GoogLeNet] Epoch [1/10], Step [400/1461], Loss: 1.7901
[GoogLeNet] Epoch [1/10], Step [500/1461], Loss: 1.8767
[GoogLeNet] Epoch [1/10], Step [600/1461], Loss: 1.4048
[GoogLeNet] Epoch [1/10], Step [700/1461], Loss: 1.9143
[GoogLeNet] Epoch [1/10], Step [800/1461], Loss: 1.7919
[GoogLeNet] Epoch [1/10], Step [900/1461], Loss: 2.1774
[GoogLeNet] Epoch [1/10], Step [1000/1461], Loss: 2.0900
[GoogLeNet] Epoch [1/10], Step [1100/1461], Loss: 2.2998
[GoogLeNet] Epoch [1/10], Step [1200/1461], Loss: 1.4501
[GoogLeNet] Epoch [1/10], Step [1300/1461], Loss: 1.6801
[GoogLeNet] Epoch [1/

...

[VGG16] Could not load pretrained weights, using random init. Reason: URLError(ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

🚀 STARTING: VGG16 | Baseline_CE (Alpha=1.0) 🚀
[VGG16] Epoch [1/10], Step [100/1461], Loss: 2.4350
[VGG16] Epoch [1/10], Step [200/1461], Loss: 2.6788
[VGG16] Epoch [1/10], Step [300/1461], Loss: 2.5473
[VGG16] Epoch [1/10], Step [400/1461], Loss: 2.5850
[VGG16] Epoch [1/10], Step [500/1461], Loss: 2.7831
[VGG16] Epoch [1/10], Step [600/1461], Loss: 2.5234
[VGG16] Epoch [1/10], Step [700/1461], Loss: 2.5311
[VGG16] Epoch [1/10], Step [800/1461], Loss: 2.2523
[VGG16] Epoch [1/10], Step [900/1461], Loss: 2.7767
[VGG16] Epoch [1/10], Step [1000/1461], Loss: 2.7496
[VGG16] Epoch [1/10], Step [1100/1461], Loss: 2.5359
[VGG16] Epoch [1/10], Step [1200/1461], Loss: 2.6079
[VGG16] Epoch [1/10], Step [1300/1461], Loss: 2.3614
[VGG16] Epoch [1/10], Step [1400/1461], Loss: 2.4726
[VGG16] End of Epoch 1 | Tra

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\Swarup/.cache\torch\hub\checkpoints\vgg16-397923af.pth


[VGG16] Could not load pretrained weights, using random init. Reason: URLError(ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

🚀 STARTING: VGG16 | Hybrid_MSE (Alpha=0.5) 🚀
[VGG16] Epoch [1/10], Step [100/1461], Loss: 11.1471
[VGG16] Epoch [1/10], Step [200/1461], Loss: 8.3971
[VGG16] Epoch [1/10], Step [300/1461], Loss: 11.6237
[VGG16] Epoch [1/10], Step [400/1461], Loss: 13.8572
[VGG16] Epoch [1/10], Step [500/1461], Loss: 13.9212
[VGG16] Epoch [1/10], Step [600/1461], Loss: 6.8340
[VGG16] Epoch [1/10], Step [700/1461], Loss: 14.1017
[VGG16] Epoch [1/10], Step [800/1461], Loss: 10.3197
[VGG16] Epoch [1/10], Step [900/1461], Loss: 8.9392
[VGG16] Epoch [1/10], Step [1000/1461], Loss: 9.1756
[VGG16] Epoch [1/10], Step [1100/1461], Loss: 14.5924
[VGG16] Epoch [1/10], Step [1200/1461], Loss: 10.2352
[VGG16] Epoch [1/10], Step [1300/1461], Loss: 10.5361
[VGG16] Epoch [1/10], Step [1400/1461], Loss: 10.5749
[VGG16] End of Epoc

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to C:\Users\Swarup/.cache\torch\hub\checkpoints\vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:22<00:00, 25.7MB/s] 



🚀 STARTING: VGG19 | Baseline_CE (Alpha=1.0) 🚀
[VGG19] Epoch [1/10], Step [100/1461], Loss: 2.2444
[VGG19] Epoch [1/10], Step [200/1461], Loss: 2.1007
[VGG19] Epoch [1/10], Step [300/1461], Loss: 1.6632
[VGG19] Epoch [1/10], Step [400/1461], Loss: 1.7336
[VGG19] Epoch [1/10], Step [500/1461], Loss: 1.8456
[VGG19] Epoch [1/10], Step [600/1461], Loss: 1.8176
[VGG19] Epoch [1/10], Step [700/1461], Loss: 1.4343
[VGG19] Epoch [1/10], Step [800/1461], Loss: 2.0910
[VGG19] Epoch [1/10], Step [900/1461], Loss: 1.6988
[VGG19] Epoch [1/10], Step [1000/1461], Loss: 1.6247
[VGG19] Epoch [1/10], Step [1100/1461], Loss: 1.7771
[VGG19] Epoch [1/10], Step [1200/1461], Loss: 1.7542
[VGG19] Epoch [1/10], Step [1300/1461], Loss: 1.5690
[VGG19] Epoch [1/10], Step [1400/1461], Loss: 1.8388
[VGG19] End of Epoch 1 | Train Tol: 62.33% | Val Tol: 66.20%
[VGG19] Epoch [2/10], Step [100/1461], Loss: 1.4463
[VGG19] Epoch [2/10], Step [200/1461], Loss: 1.5497
[VGG19] Epoch [2/10], Step [300/1461], Loss: 1.7913
[VG

In [2]:
# 1. SETUP
models_list = ["MobileNet", "GoogLeNet", "InceptionV3", "VGG16", "VGG19"]
alphas = [("Baseline_CE", "Baseline"), ("Hybrid_MSE", "Hybrid")]
output_dir = './outputs'
results = []

print("Reading tournament data...")

for model in models_list:
    for alpha_file, alpha_label in alphas:
        pth_path = os.path.join(output_dir, f"{model}_{alpha_file}.pth")
        if os.path.exists(pth_path):
            checkpoint = torch.load(pth_path, map_location='cpu', weights_only=False)
            h = checkpoint['history']
            
            # Extract metrics
            res = {
                'Model': model,
                'Loss_Type': alpha_label,
                'Val_Exact': max(h['val_exact_acc']) * 100,
                'Val_Tol': max(h['val_tol_acc']) * 100,
                'Overfit_Gap': (h['train_exact_acc'][-1] - h['val_exact_acc'][-1]) * 100,
                'Final_Train_Loss': h['train_loss'][-1],
                'Learning_Rate_Stability': np.std(h['train_loss'][-5:]) # Stability of last 5 epochs
            }
            results.append(res)

df = pd.DataFrame(results)

# 2. PLOTTING
if not df.empty:
    plt.style.use('seaborn-v0_8-muted')
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    models_present = df['Model'].unique()
    x = np.arange(len(models_present))
    width = 0.35

    def get_data(col, loss_type):
        return [df[(df['Model'] == m) & (df['Loss_Type'] == loss_type)][col].values[0] 
                if not df[(df['Model'] == m) & (df['Loss_Type'] == loss_type)].empty else 0 
                for m in models_present]

    # --- PANEL 1: IMPACT ON TOLERANCE ACCURACY ---
    axes[0, 0].bar(x - width/2, get_data('Val_Tol', 'Baseline'), width, label='Baseline (CE)', color='#1f77b4', edgecolor='black')
    axes[0, 0].bar(x + width/2, get_data('Val_Tol', 'Hybrid'), width, label='Hybrid (MSE)', color='#ff7f0e', edgecolor='black')
    axes[0, 0].set_title('Tournament Goal: Tolerance Accuracy (±1 Phase)', fontweight='bold')
    axes[0, 0].set_ylabel('Accuracy (%)')
    axes[0, 0].set_xticks(x, models_present)
    axes[0, 0].legend()

    # --- PANEL 2: OVERFITTING IMPACT ---
    axes[0, 1].bar(x - width/2, get_data('Overfit_Gap', 'Baseline'), width, label='Baseline Gap', color='#2ca02c', alpha=0.7)
    axes[0, 1].bar(x + width/2, get_data('Overfit_Gap', 'Hybrid'), width, label='Hybrid Gap', color='#d62728', alpha=0.7)
    axes[0, 1].set_title('Overfitting: Train vs Val Gap (Lower is Better)', fontweight='bold')
    axes[0, 1].set_ylabel('Gap Percentage (%)')
    axes[0, 1].set_xticks(x, models_present)
    axes[0, 1].legend()

    # --- PANEL 3: IMPACT ON EXACT ACCURACY ---
    axes[1, 0].bar(x - width/2, get_data('Val_Exact', 'Baseline'), width, label='Baseline (CE)', color='#9467bd', alpha=0.8)
    axes[1, 0].bar(x + width/2, get_data('Val_Exact', 'Hybrid'), width, label='Hybrid (MSE)', color='#bcbd22', alpha=0.8)
    axes[1, 0].set_title('Impact on Exact Classification', fontweight='bold')
    axes[1, 0].set_ylabel('Accuracy (%)')
    axes[1, 0].set_xticks(x, models_present)
    axes[1, 0].legend()

    # --- PANEL 4: LEARNING STYLE (CONVERGENCE) ---
    axes[1, 1].bar(x - width/2, get_data('Final_Train_Loss', 'Baseline'), width, label='Baseline Final Loss', color='gray')
    axes[1, 1].bar(x + width/2, get_data('Final_Train_Loss', 'Hybrid'), width, label='Hybrid Final Loss', color='brown', alpha=0.6)
    axes[1, 1].set_title('Learning Intensity: Final Loss Value', fontweight='bold')
    axes[1, 1].set_ylabel('Loss Value')
    axes[1, 1].set_xticks(x, models_present)
    axes[1, 1].legend()

    plt.suptitle("Master Tournament Analysis: Hybrid vs Baseline Loss Functions", fontsize=20, fontweight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig('./outputs/FINAL_TOURNAMENT_DASHBOARD.png', dpi=300)
    plt.show()
    print("Dashboard saved as './outputs/FINAL_TOURNAMENT_DASHBOARD.png'")
else:
    print("No data found. Ensure the tournament finished and .pth files are in ./outputs")

Reading tournament data...


NameError: name 'os' is not defined

In [2]:
print("hello")

hello


In [ ]:
import sys
import os
print('sys.executable:', sys.executable)
print('CONDA_PREFIX:', os.environ.get('CONDA_PREFIX'))
print('sys.version:', sys.version)